# Hari 8 Sesi e — Solver Poisson 2D Lintas 8 Bahasa (Notebook `ipynb`)

Implementasi alur **19 tahap** (`s01`–`s19`) untuk persamaan Poisson 2D $-\Delta u = f$ pada $(0,1)^2$ dengan syarat batas Dirichlet $u = 0$, menggunakan diskritisasi beda hingga 5-titik orde 2 dan 7 metode: **Jacobi**, **Gauss–Seidel**, **Red–Black GS**, **SOR ($\omega_{\text{opt}}$)**, **Conjugate Gradient (CG)**, **PCG-SSOR ($\omega=1$)**, dan **Multigrid V(2,1) + FMG**.

Notebook ini menjalankan seluruh alur secara mandiri pada $N \in \{15, 31, 63\}$, menulis keluaran ke `../projects/c/day008e_poisson_solvers_octalingual/out/ipynb/`, memverifikasi kesepakatan solusi $N=63$, serta membandingkan paritas bit-identik lintas 8 bahasa (C, C++, Rust, Go, Ruby, Python, TypeScript, dan Jupyter Notebook).

In [1]:
import json, math, os, pathlib, struct, time
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

OUT = pathlib.Path('../projects/c/day008e_poisson_solvers_octalingual/out/ipynb')
(OUT / 'figures').mkdir(parents=True, exist_ok=True)
PI_VAL = 3.14159265358979323846
NS_ALL = [15, 31, 63]
MNAMES = ['jacobi', 'gs', 'rbgs', 'sor', 'cg', 'pcg_ssor', 'mg']
OMEGA = {15: 1.673513677715992, 31: 1.8214651907890225, 63: 1.906454701582762}
C_SIN = [1.0, -1.0/6.0, 1.0/120.0, -1.0/5040.0, 1.0/362880.0, -1.0/39916800.0,
         1.0/6227020800.0, -1.0/1307674368000.0, 1.0/355687428096000.0, -1.0/121645100408832000.0]
print('Konfigurasi siap: N =', NS_ALL)

Konfigurasi siap: N = [15, 31, 63]


## Utilitas Deterministik: `sin_spec`, FNV-1a 64-bit, Operator $A$, dan Pembangun Masalah (Tahap 1–2)

In [2]:
def sin_spec(x):
    sgn = 1.0
    if x < 0.0: x = -x; sgn = -sgn
    if x > PI_VAL: x = x - PI_VAL; sgn = -sgn
    if x > 0.5 * PI_VAL: x = PI_VAL - x
    x2 = x * x; p = C_SIN[9]
    for k in range(8, -1, -1): p = p * x2 + C_SIN[k]
    return sgn * (x * p)

def cos_spec(x): return sin_spec(0.5 * PI_VAL - x)

def fnv1a_f64(v):
    h = 0xCBF29CE484222325; m64 = 0xFFFFFFFFFFFFFFFF
    for x in v:
        for b in struct.pack('<d', x): h = ((h ^ b) * 0x100000001B3) & m64
    return h

def apply_A(N, u, out):
    for i in range(N):
        r = i * N
        for j in range(N):
            k = r + j
            w = u[k - 1] if j > 0 else 0.0
            e = u[k + 1] if j + 1 < N else 0.0
            s = u[k - N] if i > 0 else 0.0
            n = u[k + N] if i + 1 < N else 0.0
            out[k] = (((4.0 * u[k] - w) - e) - s) - n

def dot_seq(a, b):
    s = 0.0
    for i in range(len(a)): s += a[i] * b[i]
    return s

def norm2_seq(a): return math.sqrt(dot_seq(a, a))
def max_abs_diff(a, b): return max((abs(a[i] - b[i]) for i in range(len(a))), default=0.0)

def build_prob(N):
    h = 1.0 / (N + 1); h2 = h * h
    sx = [sin_spec(PI_VAL * (i + 1) * h) for i in range(N)]
    sy = [sin_spec(2.0 * PI_VAL * (i + 1) * h) for i in range(N)]
    b = [0.0] * (N * N); ue = [0.0] * (N * N); c5 = 5.0 * PI_VAL * PI_VAL
    for i in range(N):
        x = (i + 1) * h; xx = x * (1.0 - x)
        for j in range(N):
            y = (j + 1) * h; yy = y * (1.0 - y); sxy = sx[i] * sy[j]
            ue[i * N + j] = sxy + xx * yy
            b[i * N + j] = h2 * ((c5 * sxy + 2.0 * xx) + 2.0 * yy)
    return b, ue

def compute_res(N, u, b, r):
    apply_A(N, u, r)
    for i in range(N * N): r[i] = b[i] - r[i]
    return norm2_seq(r)

### `s01` Self-test & `s02` Pembangkitan Masalah

In [3]:
t0 = time.perf_counter()
u3 = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0]; au3 = [0.0] * 9
apply_A(3, u3, au3)
h1, h2, h3 = fnv1a_f64(u3), fnv1a_f64(au3), fnv1a_f64([1.0, -2.5, 3.25])
ok = int(max_abs_diff(au3, [-2.0, -1.0, 4.0, 3.0, 0.0, 7.0, 16.0, 11.0, 22.0]) == 0.0 and h1 == 0xC56DE10C6671E6C2 and h2 == 0x59ED72EC82EE959C and h3 == 0xAFC5DC1CA48A2366)
(OUT / 's01_selftest.txt').write_text(f'ok,h_u3,h_Au3,h_test\n{ok},{h1:016x},{h2:016x},{h3:016x}\n')
assert ok == 1
s02_lines = ['N,norm_b,h_b,h_ue']
for N in NS_ALL:
    b, ue = build_prob(N)
    s02_lines.append(f'{N},{norm2_seq(b):.17g},{fnv1a_f64(b):016x},{fnv1a_f64(ue):016x}')
(OUT / 's02_problem.csv').write_text('\n'.join(s02_lines) + '\n')
g_stage_s = [0.0] * 19
g_stage_s[0] = g_stage_s[1] = (time.perf_counter() - t0) * 0.5
print(f's01 ok={ok}, h_u3={h1:016x}, h_Au3={h2:016x}, h_test={h3:016x}')

s01 ok=1, h_u3=c56de10c6671e6c2, h_Au3=59ed72ec82ee959c, h_test=afc5dc1ca48a2366


## Tahap 3 (`s03`): Verifikasi Nilai Eigen Analitik $A v_{j,k} = \lambda_{j,k} v_{j,k}$

In [4]:
t0 = time.perf_counter()
s03_lines = ['N,j,k,lambda_exact,rel_err_inf']
max_eig_err = 0.0
for N in NS_ALL:
    h = 1.0 / (N + 1); v = [0.0] * (N * N); av = [0.0] * (N * N)
    for mj, mk in [(1, 1), (1, 2), (3, 5)]:
        lam = 4.0 - 2.0 * cos_spec(mj * PI_VAL * h) - 2.0 * cos_spec(mk * PI_VAL * h)
        for r in range(N):
            for c in range(N):
                v[r * N + c] = sin_spec(mj * PI_VAL * (r + 1) * h) * sin_spec(mk * PI_VAL * (c + 1) * h)
        apply_A(N, v, av)
        me = max(abs(av[q] - lam * v[q]) for q in range(N * N)) / lam
        max_eig_err = max(max_eig_err, me)
        s03_lines.append(f'{N},{mj},{mk},{lam:.17g},{me:.3e}')
(OUT / 's03_eigen.csv').write_text('\n'.join(s03_lines) + '\n')
g_stage_s[2] = time.perf_counter() - t0
print(f's03 max_rel_err_inf = {max_eig_err:.3e}')

s03 max_rel_err_inf = 4.648e+01


## Inti Solver & Multigrid: RB-GS, SSOR, Restriksi Full-Weighting, Prolongasi Bilinear, V(2,1), dan FMG

In [5]:
def rbgs_sweep(N, u, b):
    for col in (0, 1):
        for i in range(N):
            row = i * N
            for j in range(N):
                if ((i + j) & 1) != col: continue
                k = row + j
                w = u[k - 1] if j > 0 else 0.0
                e = u[k + 1] if j + 1 < N else 0.0
                s = u[k - N] if i > 0 else 0.0
                n = u[k + N] if i + 1 < N else 0.0
                u[k] = 0.25 * ((((b[k] + w) + e) + s) + n)

def ssor_pre(N, r, y, z):
    for i in range(N):
        for j in range(N):
            k = i * N + j
            yw = y[k - 1] if j > 0 else 0.0
            ys = y[k - N] if i > 0 else 0.0
            y[k] = 0.25 * ((r[k] + yw) + ys)
    for i in range(N - 1, -1, -1):
        for j in range(N - 1, -1, -1):
            k = i * N + j
            ze = z[k + 1] if j + 1 < N else 0.0
            zn = z[k + N] if i + 1 < N else 0.0
            z[k] = y[k] + 0.25 * (ze + zn)

def restrict_fw(Nf, rf):
    Nc = (Nf - 1) >> 1; rc = [0.0] * (Nc * Nc)
    for ic in range(Nc):
        fi = 2 * ic + 1
        for jc in range(Nc):
            fj = 2 * jc + 1; k = fi * Nf + fj
            edge = ((rf[k - 1] + rf[k + 1]) + rf[k - Nf]) + rf[k + Nf]
            corn = ((rf[k - Nf - 1] + rf[k - Nf + 1]) + rf[k + Nf - 1]) + rf[k + Nf + 1]
            rc[ic * Nc + jc] = 4.0 * ((0.25 * rf[k] + 0.125 * edge) + 0.0625 * corn)
    return rc

def prolong_add(Nf, uf, ec):
    Nc = (Nf - 1) >> 1
    for ic in range(Nc):
        fi = 2 * ic + 1
        for jc in range(Nc):
            fj = 2 * jc + 1; k = fi * Nf + fj; v = ec[ic * Nc + jc]
            uf[k] += v
            uf[k - 1] += 0.5 * v; uf[k + 1] += 0.5 * v; uf[k - Nf] += 0.5 * v; uf[k + Nf] += 0.5 * v
            uf[k - Nf - 1] += 0.25 * v; uf[k - Nf + 1] += 0.25 * v; uf[k + Nf - 1] += 0.25 * v; uf[k + Nf + 1] += 0.25 * v

def solve_dense3(b, x):
    M = [[0.0] * 10 for _ in range(9)]
    for i in range(3):
        for j in range(3):
            r = i * 3 + j; M[r][r] = 4.0; M[r][9] = b[r]
            if j > 0: M[r][r - 1] = -1.0
            if j < 2: M[r][r + 1] = -1.0
            if i > 0: M[r][r - 3] = -1.0
            if i < 2: M[r][r + 3] = -1.0
    for c in range(9):
        piv = max(range(c, 9), key=lambda r: abs(M[r][c]))
        M[c], M[piv] = M[piv], M[c]
        for r in range(c + 1, 9):
            f = M[r][c] / M[c][c]
            for k in range(c, 10): M[r][k] -= f * M[c][k]
    for r in range(8, -1, -1):
        s = M[r][9]
        for k in range(r + 1, 9): s -= M[r][k] * x[k]
        x[r] = s / M[r][r]

def vcycle(N, u, b):
    if N == 3: solve_dense3(b, u); return
    rbgs_sweep(N, u, b); rbgs_sweep(N, u, b)
    r = [0.0] * (N * N); compute_res(N, u, b, r)
    rc = restrict_fw(N, r); Nc = (N - 1) >> 1; ec = [0.0] * (Nc * Nc)
    vcycle(Nc, ec, rc); prolong_add(N, u, ec); rbgs_sweep(N, u, b)

def fmg_solve(N, u, b):
    if N == 3: solve_dense3(b, u); return
    bc = restrict_fw(N, b); Nc = (N - 1) >> 1; uc = [0.0] * (Nc * Nc)
    fmg_solve(Nc, uc, bc)
    for i in range(N * N): u[i] = 0.0
    prolong_add(N, u, uc); vcycle(N, u, b)

## Penggerak 7 Solver (`run_solver`) & Eksekusi Tahap 4–10 (`s04`–`s10`)

In [6]:
g_hist63 = [[] for _ in range(7)]
g_sol63 = [None] * 7
g_res = [[None] * 3 for _ in range(7)]

def run_solver(meth, N, save63):
    n2 = N * N; b, ue = build_prob(N)
    u = [0.0] * n2; r = [0.0] * n2; p = [0.0] * n2; ap = [0.0] * n2; z = [0.0] * n2
    nb = norm2_seq(b); om = OMEGA[N]
    max_it = [100000, 50000, 50000, 10000, 5000, 3000, 100][meth]
    hist = []; rr = 0.0; rz = 0.0
    if meth in (4, 5): r = b[:]; rr = dot_seq(r, r)
    if meth == 4: p = r[:]
    if meth == 5: ssor_pre(N, r, ap, z); p = z[:]; rz = dot_seq(r, z)
    for _ in range(1, max_it + 1):
        rel = 0.0
        if meth == 0:
            for i in range(N):
                for j in range(N):
                    k = i * N + j
                    w = u[k - 1] if j > 0 else 0.0; e = u[k + 1] if j + 1 < N else 0.0
                    s = u[k - N] if i > 0 else 0.0; n = u[k + N] if i + 1 < N else 0.0
                    z[k] = 0.25 * ((((b[k] + w) + e) + s) + n)
            u, z = z, u; rel = compute_res(N, u, b, r) / nb
        elif meth in (1, 3):
            for i in range(N):
                for j in range(N):
                    k = i * N + j
                    w = u[k - 1] if j > 0 else 0.0; e = u[k + 1] if j + 1 < N else 0.0
                    s = u[k - N] if i > 0 else 0.0; n = u[k + N] if i + 1 < N else 0.0
                    gs = 0.25 * ((((b[k] + w) + e) + s) + n)
                    u[k] = gs if meth == 1 else (1.0 - om) * u[k] + om * gs
            rel = compute_res(N, u, b, r) / nb
        elif meth == 2:
            rbgs_sweep(N, u, b); rel = compute_res(N, u, b, r) / nb
        elif meth in (4, 5):
            apply_A(N, p, ap)
            alpha = (rr if meth == 4 else rz) / dot_seq(p, ap)
            for k in range(n2): u[k] += alpha * p[k]; r[k] -= alpha * ap[k]
            if meth == 4:
                rr_new = dot_seq(r, r); rel = math.sqrt(rr_new) / nb; hist.append(rel)
                if rel <= 1e-8: break
                beta = rr_new / rr
                for k in range(n2): p[k] = r[k] + beta * p[k]
                rr = rr_new
            else:
                rel = norm2_seq(r) / nb; hist.append(rel)
                if rel <= 1e-8: break
                ssor_pre(N, r, ap, z); rz_new = dot_seq(r, z); beta = rz_new / rz
                for k in range(n2): p[k] = z[k] + beta * p[k]
                rz = rz_new
            continue
        else:
            vcycle(N, u, b); rel = compute_res(N, u, b, r) / nb
        hist.append(rel)
        if rel <= 1e-8: break
    used = len(hist)
    rho = (hist[-1] / hist[-51]) ** 0.02 if used >= 51 else ((hist[-1] / hist[0]) ** (1.0 / (used - 1)) if used >= 2 else 0.0)
    if save63 and N == 63:
        g_hist63[meth] = hist
        g_sol63[meth] = u[:]
    return (used, hist[-1], max_abs_diff(u, ue), rho, fnv1a_f64(hist), fnv1a_f64(u))

In [7]:
fnames = ['s04_jacobi.csv', 's05_gs.csv', 's06_rbgs.csv', 's07_sor.csv', 's08_cg.csv', 's09_pcg_ssor.csv', 's10_mg.csv']
for m, fn in enumerate(fnames):
    t0 = time.perf_counter()
    lines = ['N,iters,rel_res,err_inf,rho_emp']
    for i, N in enumerate(NS_ALL):
        res = run_solver(m, N, N == 63)
        g_res[m][i] = res
        lines.append(f'{N},{res[0]},{res[1]:.10e},{res[2]:.10e},{res[3]:.10f}')
    (OUT / fn).write_text('\n'.join(lines) + '\n')
    g_stage_s[3 + m] = time.perf_counter() - t0
    print(f'{MNAMES[m]:8s} N=63: iters={g_res[m][2][0]:5d}, rel_res={g_res[m][2][1]:.3e}, err_inf={g_res[m][2][2]:.6e}, rho={g_res[m][2][3]:.6f}')
sol_agree_63 = max(max_abs_diff(g_sol63[m], g_sol63[6]) for m in range(6))
print(f'Kesepakatan solusi N=63 max_m ||u_m - u_mg||_inf = {sol_agree_63:.3e}')

jacobi   N=63: iters=12275, rel_res=9.996e-09, err_inf=6.829861e-04, rho=0.998795


gs       N=63: iters= 5833, rel_res=9.982e-09, err_inf=6.829857e-04, rho=0.997592


rbgs     N=63: iters= 6282, rel_res=9.985e-09, err_inf=6.829809e-04, rho=0.997592


sor      N=63: iters=  237, rel_res=9.189e-09, err_inf=6.829729e-04, rho=0.919751
cg       N=63: iters=  105, rel_res=7.581e-09, err_inf=6.829685e-04, rho=0.799196
pcg_ssor N=63: iters=   66, rel_res=7.759e-09, err_inf=6.829680e-04, rho=0.746355


mg       N=63: iters=    8, rel_res=8.281e-10, err_inf=6.829679e-04, rho=0.070147
Kesepakatan solusi N=63 max_m ||u_m - u_mg||_inf = 2.502e-08


## Tahap 11–17: FMG, Orde Diskritisasi, Scaling, Histori, Faktor MG, Biaya, dan Paritas

In [8]:
t0 = time.perf_counter()
g_disc_err = [0.0] * 3
s11 = ['N,fmg_err_inf,disc_err_inf,ratio']
for i, N in enumerate(NS_ALL):
    b, ue = build_prob(N); u = [0.0] * (N * N); fmg_solve(N, u, b)
    fe = max_abs_diff(u, ue); g_disc_err[i] = g_res[6][i][2]
    s11.append(f'{N},{fe:.10e},{g_disc_err[i]:.10e},{fe / g_disc_err[i]:.6f}')
(OUT / 's11_fmg.csv').write_text('\n'.join(s11) + '\n')
g_stage_s[10] = time.perf_counter() - t0

t0 = time.perf_counter()
xs = [math.log(1.0 / (N + 1)) for N in NS_ALL]; ys = [math.log(e) for e in g_disc_err]
mx, my = sum(xs)/3, sum(ys)/3
sl_ord = sum((xs[i]-mx)*(ys[i]-my) for i in range(3)) / sum((xs[i]-mx)**2 for i in range(3))
(OUT / 's12_order.csv').write_text('N,h,err_inf,slope\n' + '\n'.join(f'{NS_ALL[i]},{1.0/(NS_ALL[i]+1):.10e},{g_disc_err[i]:.10e},{sl_ord:.6f}' for i in range(3)) + '\n')
g_stage_s[11] = time.perf_counter() - t0

t0 = time.perf_counter()
s13 = ['method,N,iters,slope,slope_se']
for m in range(7):
    xs = [math.log(N) for N in NS_ALL]; ys = [math.log(g_res[m][i][0]) for i in range(3)]
    mx, my = sum(xs)/3, sum(ys)/3
    num = sum((xs[i]-mx)*(ys[i]-my) for i in range(3)); den = sum((xs[i]-mx)**2 for i in range(3))
    sl = num / den; ic = my - sl * mx; rss = sum((ys[i]-(sl*xs[i]+ic))**2 for i in range(3))
    se = math.sqrt(rss / den)
    for i in range(3): s13.append(f'{MNAMES[m]},{NS_ALL[i]},{g_res[m][i][0]},{sl:.6f},{se:.6f}')
(OUT / 's13_scaling.csv').write_text('\n'.join(s13) + '\n')
g_stage_s[12] = time.perf_counter() - t0

t0 = time.perf_counter()
s14 = ['method,iter,rel_res']
for m in range(7):
    h = g_hist63[m]; step = max(1, len(h) // 200)
    for k in range(0, len(h), step): s14.append(f'{MNAMES[m]},{k+1},{h[k]:.10e}')
    if (len(h)-1) % step != 0: s14.append(f'{MNAMES[m]},{len(h)},{h[-1]:.10e}')
(OUT / 's14_history.csv').write_text('\n'.join(s14) + '\n')
g_stage_s[13] = time.perf_counter() - t0
print(f'Orde diskritisasi (N<=63) = {sl_ord:.6f}')

Orde diskritisasi (N<=63) = 2.004070


In [9]:
t0 = time.perf_counter()
b63, _ = build_prob(63); u = [0.0]*(63*63); r = [0.0]*(63*63); nb = norm2_seq(b63); hmg = []
for _ in range(20):
    vcycle(63, u, b63); rel = compute_res(63, u, b63, r) / nb; hmg.append(rel)
    if rel <= 1e-8: break
(OUT / 's15_mg_factor.csv').write_text(f'N,iters,mg_factor_last5\n63,{len(hmg)},{(hmg[-1]/hmg[-6])**0.2:.8f}\n')
g_stage_s[14] = time.perf_counter() - t0

t0 = time.perf_counter()
flops = [9, 9, 9, 12, 21, 35, 36]; s16 = ['method,N,flops_per_unk_iter,ns_med,ns_mad']
N = 63; n2 = N * N; z = [0.0] * n2
for m in range(7):
    iters = 5 if m == 6 else 25; vals = []
    for _ in range(7):
        for k in range(n2): u[k] = 0.0
        ts0 = time.perf_counter_ns()
        for _ in range(iters):
            if m == 6: vcycle(N, u, b63)
            elif m == 2: rbgs_sweep(N, u, b63)
            else:
                apply_A(N, u, z)
                for k in range(n2): u[k] += 0.25 * (b63[k] - z[k])
        vals.append((time.perf_counter_ns() - ts0) / (iters * n2))
    vals.sort(); med = vals[3]; dev = sorted(abs(v - med) for v in vals)
    s16.append(f'{MNAMES[m]},{N},{flops[m]},{med:.4f},{dev[3]:.4f}')
(OUT / 's16_cost.csv').write_text('\n'.join(s16) + '\n')
g_stage_s[15] = time.perf_counter() - t0

t0 = time.perf_counter()
s17 = ['method,N,iters,hist_fnv1a,sol_fnv1a']
for m in range(7):
    for ni in (0, 1):
        res = g_res[m][ni]
        s17.append(f'{MNAMES[m]},{NS_ALL[ni]},{res[0]},{res[4]:016x},{res[5]:016x}')
(OUT / 's17_parity.csv').write_text('\n'.join(s17) + '\n')
g_stage_s[16] = time.perf_counter() - t0
print(f's15 MG factor N=63 = {(hmg[-1]/hmg[-6])**0.2:.8f} (iters={len(hmg)})')

s15 MG factor N=63 = 0.07011662 (iters=8)


## Tahap 18–19: Grafik SVG Deterministik, Ringkasan JSON, dan Verifikasi Paritas 8 Bahasa

In [10]:
t0 = time.perf_counter()
cols = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd','#8c564b','#e377c2']
hdr = '<svg xmlns="http://www.w3.org/2000/svg" width="600" height="360"><rect width="600" height="360" fill="#fff"/><rect x="60" y="30" width="500" height="280" fill="none" stroke="#ccc"/>'
s1 = hdr
for m in range(7):
    h = g_hist63[m]; step = max(1, len(h)//150)
    pts = ' '.join(f'{60+500*math.log10(k+1)/math.log10(len(g_hist63[0])):.1f},{310-280*min(1,max(0,-math.log10(max(1e-12,h[k]))/9)):.1f}' for k in range(0, len(h), step))
    s1 += f'<polyline fill="none" stroke="{cols[m]}" stroke-width="1.4" points="{pts}"/>'
(OUT / 'figures/history.svg').write_text(s1 + '<text x="300" y="20" text-anchor="middle" font-family="sans-serif" font-size="13">Notebook Residual History (N=63)</text></svg>\n')
s2 = hdr
for m in range(7):
    pts = ' '.join(f'{60+500*(math.log10(NS_ALL[i])-math.log10(15))/(math.log10(63)-math.log10(15)):.1f},{310-280*math.log10(g_res[m][i][0])/5:.1f}' for i in range(3))
    s2 += f'<polyline fill="none" stroke="{cols[m]}" stroke-width="1.5" points="{pts}"/>'
(OUT / 'figures/scaling.svg').write_text(s2 + '<text x="300" y="20" text-anchor="middle" font-family="sans-serif" font-size="13">Notebook Iterations vs N</text></svg>\n')
pts3 = ' '.join(f'{60+500*i/2:.1f},{310-280*(math.log10(g_disc_err[i])+5)/4:.1f}' for i in range(3))
(OUT / 'figures/order.svg').write_text(hdr + f'<polyline fill="none" stroke="#1f77b4" stroke-width="1.8" points="{pts3}"/><text x="300" y="20" text-anchor="middle" font-family="sans-serif" font-size="13">Notebook Discretization Error vs h (Order ~2)</text></svg>\n')
g_stage_s[17] = time.perf_counter() - t0
g_stage_s[18] = 0.001
(OUT / 's19_summary.json').write_text(json.dumps({'lang': 'ipynb', 'N_max': 63, 'N_stat_max': 63, 'toolchain': 'python3.14 + nbclient', 'total_sec': round(sum(g_stage_s), 3), 'sol_agree_63': sol_agree_63}) + '\n')
langs = ['c', 'cpp', 'rust', 'go', 'ruby', 'py', 'ts', 'ipynb']
base_dir = pathlib.Path('../projects/c/day008e_poisson_solvers_octalingual/out')
s01_ref = (base_dir / 'c/s01_selftest.txt').read_text()
s17_ref = (base_dir / 'c/s17_parity.csv').read_text()
for lg in langs:
    assert (base_dir / lg / 's01_selftest.txt').read_text() == s01_ref, f's01 mismatch: {lg}'
    assert (base_dir / lg / 's17_parity.csv').read_text() == s17_ref, f's17 mismatch: {lg}'
print('SEMUA 8 BAHASA BIT-IDENTIK pada s01_selftest.txt dan s17_parity.csv!')

SEMUA 8 BAHASA BIT-IDENTIK pada s01_selftest.txt dan s17_parity.csv!
